In [ ]:
# ! 計算總體

from scipy.stats import entropy
import numpy as np

probabilities = np.array([1/6] * 6)

# 注意：必須指定 base=2 才會算出位元(bits)，否則預設會使用自然對數(base=e)
result = entropy(probabilities, base=2)

print(f"SciPy 計算結果: {result:.4f} bits")

SciPy 計算結果: 2.5850 bits


In [ ]:
# ! 計算個體

import numpy as np

# 模型的預測機率
probs = np.array([0.95, 0.03, 0.02])
classes = ['貓', '狗', '鳥']

# 避免 log2(0) 的數學錯誤，給機率加上極小值 epsilon
epsilon = 1e-10
safe_probs = np.clip(probs, epsilon, 1.0)

# 直接利用 numpy 陣列計算單項貢獻度：-p * log2(p)
# 這裡算出來的會是一個陣列：[0.0703, 0.1518, 0.1129]
contributions = -safe_probs * np.log2(safe_probs)

# 印出每個類別的貢獻度
for cls, p, c in zip(classes, probs, contributions):
    print(f"{cls} (機率 {p:.2f}) 的熵貢獻度: {c:.4f} bits")

# 驗證總和
total_entropy = np.sum(contributions)
print(f"-> 總體熵: {total_entropy:.4f} bits")

貓 (機率 0.95) 的熵貢獻度: 0.0703 bits
狗 (機率 0.03) 的熵貢獻度: 0.1518 bits
鳥 (機率 0.02) 的熵貢獻度: 0.1129 bits
-> 總體熵: 0.3349 bits


----
----
----

In [3]:
import numpy as np

probs = np.array([0.95, 0.03, 0.02])
gini_index = 1 - np.sum(probs**2)

print(f"吉尼係數為: {gini_index:.4f}")

吉尼係數為: 0.0962


----
----
----

In [5]:
import pandas as pd
import numpy as np
from sklearn.metrics import mutual_info_score
from sklearn.tree import DecisionTreeClassifier

# ==========================================
# 1. 建立原始資料集 (DataFrame)
# ==========================================
# 總共 14 天
# 濕度 High 有 7 天 (3天打球, 4天不打)
# 濕度 Normal 有 7 天 (6天打球, 1天不打)

data = {
    'Humidity': ['High']*7 + ['Normal']*7,
    'PlayTennis': ['Yes']*3 + ['No']*4 + ['Yes']*6 + ['No']*1
}
df = pd.DataFrame(data)

print("=== 1. 使用 sklearn 計算 Information Gain ===")
# sklearn 使用的 mutual_info_score 底層就是 Information Gain
# 注意：它預設使用自然對數 (e)，所以我們必須除以 np.log(2) 轉成 bits
ig = mutual_info_score(df['PlayTennis'], df['Humidity']) / np.log(2)

print(f"套件算出的 Information Gain: {ig:.3f} bits")
print("(這與我們手算的 0.151 bits 完全吻合！)\n")


print("=== 2. 使用 sklearn 提取 Gini Index ===")
# sklearn 的決策樹需要餵入數字，所以我們先把文字轉成數字 (0 和 1)
X = df[['Humidity']].replace({'High': 1, 'Normal': 0})
y = df['PlayTennis'].replace({'Yes': 1, 'No': 0})

# 建立一棵深度只有 1 的決策樹 (因為我們只切一刀)
# criterion 預設就是 'gini'
clf = DecisionTreeClassifier(criterion='gini', max_depth=1, random_state=42)
clf.fit(X, y)

# ------------------------------------------
# 偷看 sklearn 黑盒子裡面算出來的 Gini 數值
# clf.tree_ 儲存了整棵樹的節點資訊
# ------------------------------------------
n_samples = clf.tree_.n_node_samples # 每個節點的資料筆數
impurities = clf.tree_.impurity      # 每個節點的 Gini 不純度

root_gini = impurities[0]   # 切分前 (14筆資料)
left_gini = impurities[1]   # 左子節點 (Normal 濕度, 7筆)
right_gini = impurities[2]  # 右子節點 (High 濕度, 7筆)

# 手動將左右子節點加權平均
weighted_gini = (n_samples[1] / n_samples[0]) * left_gini + \
                (n_samples[2] / n_samples[0]) * right_gini

print(f"套件算出的 切分前 Gini (14天): {root_gini:.3f}")
print(f"套件算出的 左節點 Gini (Normal): {left_gini:.3f}")
print(f"套件算出的 右節點 Gini (High): {right_gini:.3f}")
print(f"👉 套件算出的 加權 Gini 總和: {weighted_gini:.3f}")
print("(這與我們手算的 0.367 完全吻合！)")

=== 1. 使用 sklearn 計算 Information Gain ===
套件算出的 Information Gain: 0.152 bits
(這與我們手算的 0.151 bits 完全吻合！)

=== 2. 使用 sklearn 提取 Gini Index ===
套件算出的 切分前 Gini (14天): 0.459
套件算出的 左節點 Gini (Normal): 0.245
套件算出的 右節點 Gini (High): 0.490
👉 套件算出的 加權 Gini 總和: 0.367
(這與我們手算的 0.367 完全吻合！)


/var/folders/s0/f874234s21s4w3fp3rnc3rfm0000gn/T/ipykernel_45365/2196338082.py:30: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X = df[['Humidity']].replace({'High': 1, 'Normal': 0})
/var/folders/s0/f874234s21s4w3fp3rnc3rfm0000gn/T/ipykernel_45365/2196338082.py:31: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y = df['PlayTennis'].replace({'Yes': 1, 'No': 0})


----
----
----